In [1]:
import pickle
from utils.hebbian_neural_net import *

import io, torch

In [2]:
# force any tensor inside the pickle to land on CPU
torch.storage._load_from_bytes = lambda b: torch.load(
    io.BytesIO(b), map_location="cpu", weights_only=False
)

model_path = r"C:\Users\emper\Desktop\ICRA2026\sim2real\pos_act0_model_99.pickle"
with open(model_path, "rb") as file:
    loaded_data = pickle.load(file)

In [17]:
loaded_model = loaded_data[1]
solver = loaded_data[0]

In [ ]:
# flat_model_param = loaded_model.get_a_model_params()
flat_model_param = solver.best_param()     # (71360,) 
architecture = loaded_model.architecture

print(architecture)
print(flat_model_param.shape)


[38, 128, 64, 19]
(71360,)


In [ ]:
temp_hebb = HebbianNet(popsize=1, 
                       sizes=architecture, 
                       init_noise=0.04, 
                       norm_mode='var', 
                       std_init_param=0.025
                       )
temp_hebb.set_a_model_params(flat_model_param)


In [37]:
import json
import numpy as np

json_path = r"C:\Users\emper\Desktop\ICRA2026\sim2real\pos_act0_model_99.json"

payload = {
    "architecture": list(map(int, architecture)),
    "n_params": int(np.asarray(flat_model_param).size),
    "norm_mode": "var",
    "std_init_param": float(loaded_model.std_init_param),
    "param_order": ["A", "B", "C", "D", "lr"],
    "dtype": "float64",
    "source": "pos_act0_model_99.pickle -> solver.best_param()",
    "flat_model_param": np.asarray(flat_model_param, dtype=np.float64).tolist(),
}

with open(json_path, "w") as f:
    json.dump(payload, f)

print("saved:", json_path)

saved: C:\Users\emper\Desktop\ICRA2026\sim2real\pos_act0_model_99.json


In [38]:
# load back from JSON -- no pickle, no torch.load patch needed
with open(json_path) as f:
    cfg = json.load(f)

params = np.asarray(cfg["flat_model_param"], dtype=np.float64)

hebb = HebbianNet(popsize=1,
                  sizes=cfg["architecture"],
                  init_noise=0.04,          # TODO: HEBB_init_wnoise from the run config
                  norm_mode=cfg["norm_mode"],
                  std_init_param=cfg["std_init_param"])
hebb.set_a_model_params(params)

print("match:", np.array_equal(hebb.get_a_model_params(), np.asarray(flat_model_param)))

match: True
